# Ejercicio 7: Bases de Datos Vectoriales

**Nombre:** Andrés Pérez

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

In [19]:
!pip install pymilvus
!pip install qdrant-client
!pip install weaviate-client chromadb psycopg2-binary

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [3]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

100%|██████████| 50.6M/50.6M [00:02<00:00, 25.4MB/s]


,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [4]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [5]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [6]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [7]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/4944 [00:00<?, ?it/s]

In [8]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [9]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [10]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 76.9 MB/s eta 0:00:00


In [11]:
import faiss
import numpy as np

# Asumiendo `embeddings` en un array NxD
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# Cambiamos query_embedding por query_vec
D, I = index.search(query_vec, k=10)

print("Índices de los documentos más cercanos:", I)
print("Distancias L2:", D)

Índices de los documentos más cercanos: [[10176     1 10177 37406 71872 37409 10481     5 75249 47064]]
Distancias L2: [[0.25930297 0.27639928 0.3197968  0.32173342 0.32282233 0.33097768
  0.3313005  0.336756   0.3395679  0.34671992]]


## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [12]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import time

# Conectar (Modo in-memory)
client = QdrantClient(location=":memory:")

# Crear Colección
COLLECTION_NAME = "wikipedia_chunks"
D = embeddings.shape[1]

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=D, distance=Distance.COSINE),
)

# Insertar datos en lotes
BATCH_SIZE = 512
points = []

print("Insertando vectores en Qdrant... (esto puede tomar unos segundos)")
for i in range(len(embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=embeddings[i].tolist(),
            payload={"text": chunks_df["text"].iloc[i], "doc_id": int(chunks_df["doc_id"].iloc[i])}
        )
    )
    # Cuando llegamos a 512, subimos el lote y vaciamos la lista
    if len(points) == BATCH_SIZE:
        client.upsert(collection_name=COLLECTION_NAME, points=points)
        points = []

# Subimos los últimos que hayan quedado en la lista
if points:
    client.upsert(collection_name=COLLECTION_NAME, points=points)

print(f"Indexados {len(embeddings)} vectores exitosamente.")

def qdrant_search(query_embedding, k=5):
    t0 = time.time()
    res = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding[0].tolist(),
        limit=k
    ).points
    t1 = time.time()

    out = [(r.id, r.score, r.payload["text"]) for r in res]
    return out, (t1 - t0)

# Ejecutar la búsqueda
resultados, tiempo = qdrant_search(query_vec, k=5)
print(f"\nTiempo de búsqueda: {tiempo:.4f} segundos")

for r in resultados:
    print(f"ID: {r[0]:>5} | Score: {r[1]:.4f} | Texto: {r[2][:80]}...")

Insertando vectores en Qdrant... (esto puede tomar unos segundos)


/tmp/ipykernel_2087/1808348692.py:35: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(collection_name=COLLECTION_NAME, points=points)


Indexados 79104 vectores exitosamente.

Tiempo de búsqueda: 0.3113 segundos
ID: 10176 | Score: 0.8703 | Texto: Battery tester A battery tester is an electronic device intended for testing the...
ID:     1 | Score: 0.8618 | Texto: Battery indicator A battery indicator (also known as a battery gauge) is a devic...
ID: 10177 | Score: 0.8401 | Texto: ing procedure, according to the type of battery being tested, such as the â€œ421...
ID: 37406 | Score: 0.8391 | Texto: ils. One was connected via a series resistor to the battery supply. The second w...
ID: 71872 | Score: 0.8386 | Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C c...


## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [13]:
!pip install milvus-lite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 10.9 MB/s eta 0:00:00


In [14]:
from pymilvus import MilvusClient
import time

client = MilvusClient("milvus_demo.db")
COLLECTION_NAME = "wiki_milvus"
D = embeddings.shape[1]

# Crear colección
if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)

# Al poner dimension, Milvus Lite crea un índice básico automático
client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=D,
    metric_type="COSINE",
    id_type="int",
)

# Insertar en LOTES
BATCH_SIZE = 1000
print("Insertando vectores en Milvus por lotes...")

for i in range(0, len(embeddings), BATCH_SIZE):
    end_idx = min(i + BATCH_SIZE, len(embeddings))
    batch_data = [
        {"id": j, "vector": embeddings[j].tolist(), "text": chunks_df["text"].iloc[j]}
        for j in range(i, end_idx)
    ]
    client.insert(collection_name=COLLECTION_NAME, data=batch_data)

print(f"Indexados {len(embeddings)} vectores exitosamente.")

# Índices (ANN - HNSW)
print("Creando índice HNSW personalizado...")

try:
    client.release_collection(collection_name=COLLECTION_NAME)
    client.drop_index(collection_name=COLLECTION_NAME, index_name="vector")
except Exception as e:
    pass

index_params = client.prepare_index_params()
index_params.add_index(
    field_name="vector",
    index_type="HNSW",
    metric_type="COSINE",
    params={"M": 16, "efConstruction": 64}
)
client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

# Volvemos a cargar la colección en memoria RAM para poder buscar
client.load_collection(COLLECTION_NAME)

# Función de búsqueda
def milvus_search(query_embedding, k=5):
    t0 = time.time()
    res = client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding[0].tolist()],
        limit=k,
        search_params={"metric_type": "COSINE", "params": {"ef": 64}},
        output_fields=["text"]
    )
    t1 = time.time()
    return res, (t1 - t0)

res_milvus, t_milvus = milvus_search(query_vec, k=5)
print(f"\nTiempo de búsqueda en Milvus: {t_milvus:.4f} segundos")

for hits in res_milvus:
    for hit in hits:
        print(f"ID: {hit['id']:>5} | Score: {hit['distance']:.4f} | Texto: {hit['entity']['text'][:80]}...")

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1232, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Insertando vectores en Milvus por lotes...
Indexados 79104 vectores exitosamente.
Creando índice HNSW personalizado...


ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1232, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!



Tiempo de búsqueda en Milvus: 3.8051 segundos
ID: 10176 | Score: 0.1297 | Texto: Battery tester A battery tester is an electronic device intended for testing the...
ID:     1 | Score: 0.1382 | Texto: Battery indicator A battery indicator (also known as a battery gauge) is a devic...
ID: 10177 | Score: 0.1599 | Texto: ing procedure, according to the type of battery being tested, such as the â€œ421...
ID: 71872 | Score: 0.1614 | Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C c...
ID: 37409 | Score: 0.1655 | Texto: shorting the measurement points together and performing an adjustment for zero o...


## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [15]:
!pip install weaviate-client chromadb psycopg2-binary

In [16]:
import weaviate
from weaviate.classes.config import Property, DataType as WDataType
import time

try:
    print("Iniciando Weaviate Embebido...")
    print("(La primera vez tardará unos segundos extra porque descargará el binario de Weaviate)")

    # Se usa connect_to_embedded y cambiamos los puertos para evitar el choque
    client = weaviate.connect_to_embedded(port=8090, grpc_port=50052)

    # Crear colección
    client.collections.delete("DocumentChunk")
    collection = client.collections.create(
        name="DocumentChunk",
        properties=[
            Property(name="text", data_type=WDataType.TEXT),
            Property(name="doc_id", data_type=WDataType.INT),
        ]
    )

    # Insertar vectores
    print("Insertando vectores...")
    with collection.batch.dynamic() as batch:
        for i in range(len(embeddings)):
            batch.add_object(
                properties={"text": chunks_df["text"].iloc[i], "doc_id": int(chunks_df["doc_id"].iloc[i])},
                vector=embeddings[i].tolist()
            )

    print("Inserción completada en Weaviate exitosamente.")

    # Búsqueda de prueba para comprobar que funciona
    print("\nBuscando Top-5 resultados...")
    t0 = time.time()
    res = collection.query.near_vector(
        near_vector=query_vec[0].tolist(),
        limit=5,
        return_metadata=weaviate.classes.query.MetadataQuery(distance=True)
    )
    t1 = time.time()

    print(f"Tiempo de búsqueda: {t1 - t0:.4f} segundos")
    for r in res.objects:
        similitud = 1 - r.metadata.distance
        print(f"Score: {similitud:.4f} | Texto: {r.properties['text'][:80]}...")

    client.close()

except Exception as e:
    print(f"\n[AVISO]: Falló Weaviate Embebido ({type(e).__name__}).")
    print("Si estás usando Windows puro, Weaviate Embebido no es compatible.")

INFO:weaviate-client:Binary /root/.cache/weaviate-embedded did not exist. Downloading binary from https://github.com/weaviate/weaviate/releases/download/v1.30.5/weaviate-v1.30.5-Linux-amd64.tar.gz


Iniciando Weaviate Embebido...
(La primera vez tardará unos segundos extra porque descargará el binario de Weaviate)


INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 7828


Insertando vectores...
Inserción completada en Weaviate exitosamente.

Buscando Top-5 resultados...
Tiempo de búsqueda: 0.0068 segundos
Score: 0.8703 | Texto: Battery tester A battery tester is an electronic device intended for testing the...
Score: 0.8618 | Texto: Battery indicator A battery indicator (also known as a battery gauge) is a devic...
Score: 0.8401 | Texto: ing procedure, according to the type of battery being tested, such as the â€œ421...
Score: 0.8391 | Texto: ils. One was connected via a series resistor to the battery supply. The second w...
Score: 0.8386 | Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C c...


## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [20]:
import chromadb
import time

# Conectar y crear colección
client = chromadb.Client()
COLLECTION_NAME = "wiki_chroma"

try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(name=COLLECTION_NAME)

# Insertar en LOTES para evitar el colapso de memoria
BATCH_SIZE = 5000
print("Insertando vectores en Chroma por lotes...")

for i in range(0, len(embeddings), BATCH_SIZE):
    end_idx = min(i + BATCH_SIZE, len(embeddings))

    collection.add(
        ids=[str(j) for j in range(i, end_idx)],
        embeddings=embeddings[i:end_idx].tolist(),
        documents=chunks_df["text"].iloc[i:end_idx].tolist(),
        metadatas=[{"doc_id": int(d)} for d in chunks_df["doc_id"].iloc[i:end_idx]]
    )

print(f"Indexados {collection.count()} vectores en Chroma exitosamente.")

# Búsqueda
def chroma_search(query_embedding, k=5):
    t0 = time.time()
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k
    )
    t1 = time.time()
    return results, (t1 - t0)

res_chroma, t_chroma = chroma_search(query_vec, k=5)
print(f"\nTiempo de búsqueda en Chroma: {t_chroma:.4f} segundos")

print("\nTop-5 Resultados:")
for i in range(len(res_chroma['ids'][0])):
    doc_id = res_chroma['ids'][0][i]
    distancia = res_chroma['distances'][0][i]
    texto = res_chroma['documents'][0][i]

    print(f"ID: {doc_id:>5} | Distancia (L2/Coseno): {distancia:.4f} | Texto: {texto[:80]}...")

Insertando vectores en Chroma por lotes...
Indexados 79104 vectores en Chroma exitosamente.

Tiempo de búsqueda en Chroma: 0.0035 segundos

Top-5 Resultados:
ID: 10176 | Distancia (L2/Coseno): 0.2593 | Texto: Battery tester A battery tester is an electronic device intended for testing the...
ID:     1 | Distancia (L2/Coseno): 0.2764 | Texto: Battery indicator A battery indicator (also known as a battery gauge) is a devic...
ID: 10177 | Distancia (L2/Coseno): 0.3198 | Texto: ing procedure, according to the type of battery being tested, such as the â€œ421...
ID: 37406 | Distancia (L2/Coseno): 0.3217 | Texto: ils. One was connected via a series resistor to the battery supply. The second w...
ID: 71872 | Distancia (L2/Coseno): 0.3228 | Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C c...


## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


In [23]:
%%bash
# Instalar PostgreSQL y las herramientas de desarrollo necesarias
apt-get update > /dev/null
apt-get install -y postgresql postgresql-contrib postgresql-server-dev-14 > /dev/null

# Descargar y compilar pgvector desde su código fuente oficial (la forma más segura en Colab)
cd /tmp
rm -rf pgvector
git clone --branch v0.6.0 https://github.com/pgvector/pgvector.git > /dev/null 2>&1
cd pgvector
make > /dev/null 2>&1
make install > /dev/null 2>&1

# Iniciar el servidor de base de datos
service postgresql start

# Configurar el usuario postgres
sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'root';" > /dev/null

# Limpiar si ya existía y crear la base de datos "vectordb"
sudo -u postgres dropdb vectordb > /dev/null 2>&1 || true
sudo -u postgres createdb vectordb > /dev/null

echo "✅ PostgreSQL y pgvector instalados y corriendo correctamente en Colab."

 * Starting PostgreSQL 14 database server
   ...done.
✅ PostgreSQL y pgvector instalados y corriendo correctamente en Colab.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
createdb: error: database creation failed: ERROR:  database "vectordb" already exists


In [24]:
import psycopg2
import time

try:
    print("Conectando a PostgreSQL en Colab...")
    conn = psycopg2.connect("dbname=vectordb user=postgres password=root host=localhost")
    conn.autocommit = True
    cur = conn.cursor()

    # Crear extensión y tabla
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("DROP TABLE IF EXISTS chunks;")

    # D es la dimensión de tus embeddings de E5 (probablemente 768)
    D = embeddings.shape[1]
    cur.execute(f"CREATE TABLE chunks (id serial PRIMARY KEY, text TEXT, embedding vector({D}));")
    print("Tabla y extensión preparadas.")

    # Insertar vectores (usamos una rebanada pequeña de 1000 para que sea rápido en la demo)
    print("Insertando muestra de 1000 vectores...")
    from psycopg2.extras import execute_values

    # Preparamos los datos
    data_tuples = [
        (chunks_df["text"].iloc[i], embeddings[i].tolist())
        for i in range(1000)
    ]

    # Inserción rápida en bloque
    execute_values(
        cur,
        "INSERT INTO chunks (text, embedding) VALUES %s",
        data_tuples,
        template="(%s, %s::vector)"
    )

    # Búsqueda con SQL
    print("\nBuscando Top-5 resultados con SQL...")
    vec_str = str(query_vec[0].tolist())

    t0 = time.time()
    cur.execute(f"""
        SELECT id, text, 1 - (embedding <=> '{vec_str}') as similitud
        FROM chunks
        ORDER BY embedding <=> '{vec_str}'
        LIMIT 5;
    """)
    resultados = cur.fetchall()
    t1 = time.time()

    print(f"Tiempo de búsqueda: {t1 - t0:.4f} segundos\n")
    for r in resultados:
        print(f"ID: {r[0]:>5} | Score: {r[2]:.4f} | Texto: {r[1][:80]}...")

    cur.close()
    conn.close()

except Exception as e:
    print(f"Error: {e}")

Conectando a PostgreSQL en Colab...
Tabla y extensión preparadas.
Insertando muestra de 1000 vectores...

Buscando Top-5 resultados con SQL...
Tiempo de búsqueda: 0.0081 segundos

ID:     2 | Score: 0.8618 | Texto: Battery indicator A battery indicator (also known as a battery gauge) is a devic...
ID:     6 | Score: 0.8316 | Texto: otective diodes cannot be used, a battery will simply destroy the diodes and dam...
ID:     3 | Score: 0.8222 | Texto: ad battery when in reality it indicates a problem with the vehicle's charging sy...
ID:    15 | Score: 0.8174 | Texto: Capacity loss Capacity loss or capacity fading is a phenomenon observed in recha...
ID:     5 | Score: 0.8081 | Texto: increase; in many cases the EMF remains more or less constant during most of the...
